### **CHURN PREDICTION SYSTEM: TECHNICAL OVERVIEW**

This document outlines the architecture and methodology of the churn prediction pipeline. The system is designed to categorize users into three distinct states: active, voluntary churn (user-initiated), and involuntary churn (payment-related failure).

---

### **1. Temporal Robustness and Out-of-Time Safety**
The primary objective of the architecture is to ensure model stability when applied to future datasets. To achieve this, the pipeline utilizes a "Relative Timing" strategy. All absolute timestamps are converted into durations measured from the moment of account creation. By removing specific calendar dates, the model is forced to learn universal behavioral patterns rather than temporary trends, preventing the "leakage" of time-specific data into the prediction phase.

### **2. Multi-Source Feature Engineering**
The system synthesizes data from several operational domains to create a holistic user profile:
* **Onboarding Efficiency:** The pipeline tracks the "Time to First Success," identifying the interval between registration and the user’s first successful AI generation. This is a critical indicator of long-term retention.
* **Transactional Risk Analysis:** Beyond simple success or failure, the model analyzes "3DS Friction" (authentication hurdles) and geographical mismatches between card issuers and billing addresses to identify high-risk payment profiles.
* **Engagement Velocity:** The system calculates the rate of credit consumption and identifies steep drops in activity between the first and second weeks of usage.

### **3. Ensemble Modeling Strategy**
The predictive engine utilizes a "Soft-Voting Ensemble" combining two industry-standard gradient boosting algorithms: LightGBM and XGBoost. 
* **LightGBM** is employed for its efficiency in detecting complex, non-linear relationships within large datasets.
* **XGBoost** provides structural stability, reducing the risk of overfitting to noise in the training data.

### **4. Diagnostic Explainability Engine**
To ensure the model’s outputs are actionable for business stakeholders, a custom explainability module was developed. For every predicted churn event, the engine generates a report detailing the specific drivers of the risk. It distinguishes between technical payment issues and lack of product engagement, allowing for targeted intervention strategies rather than generic outreach.


In [ ]:
# ==============================================================================
# CHURN PREDICTION — FULL PIPELINE v2 (OOT-safe, leak-free)
# ==============================================================================
# CELL 1: Imports & Paths
# ==============================================================================
import pandas as pd
import numpy as np
import warnings
import gc
from pathlib import Path
from tqdm.notebook import tqdm
 
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
import category_encoders as ce
import lightgbm as lgb
 
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)
 
TRAIN_DIR = Path('/kaggle/input/datasets/karnakbaevarthur/hack-dataset/'
                 'drive-download-20260404T091938Z-1-001/Train Data')
TEST_DIR  = Path('/kaggle/input/datasets/karnakbaevarthur/hack-dataset/'
                 'drive-download-20260404T091938Z-1-001/Test Data')
GEN_TRAIN = ('/kaggle/input/datasets/karnakbaevarthur/hack-dataset/'
             'train_users_generations.csv/train_users_generations.csv')
GEN_TEST  = str(TEST_DIR / 'test_users_generations.csv')
 
SEED = 42
print('✅ Libraries imported and paths defined')

In [ ]:
# ==============================================================================
# CELL 2: Date Parsing Utility, Data Loaders, Subscription Windows
# ==============================================================================
 
def parse_dates(series: pd.Series) -> pd.Series:
    """
    Dates are synthetically shifted (year 1067 → 2023 after regex replace).
    This preserves month/day/time ordering — all relative features remain valid.
    We never use the absolute year, month, or calendar weekday as a model feature
    because the hidden test is OOT (different calendar period).
    """
    fixed = series.astype(str).str.replace(r'^\d{4}', '2023', regex=True)
    return pd.to_datetime(fixed, errors='coerce', utc=True)
 
 
def load_tables(base_dir: Path, prefix: str) -> dict:
    files = {
        'users':        f'{prefix}_users.csv',
        'properties':   f'{prefix}_users_properties.csv',
        'quizzes':      f'{prefix}_users_quizzes.csv',
        'purchases':    f'{prefix}_users_purchases.csv',
        'transactions': f'{prefix}_users_transaction_attempts_v1.csv',
    }
    tables = {}
    for name, filename in files.items():
        path = base_dir / filename
        if path.exists():
            tables[name] = pd.read_csv(path, low_memory=False)
            print(f'  {name}: {tables[name].shape}')
        else:
            print(f'  ⚠ Missing: {path}')
    return tables
 
 
print('Loading TRAIN...')
train = load_tables(TRAIN_DIR, 'train')
print('\nLoading TEST...')
test = load_tables(TEST_DIR, 'test')
 
 
def build_sub_dates(props_df: pd.DataFrame) -> pd.DataFrame:
    df = props_df[['user_id', 'subscription_start_date']].copy()
    df['sub_start']   = parse_dates(df['subscription_start_date'])
    df['sub_end_14d'] = df['sub_start'] + pd.Timedelta(days=14)
    return df[['user_id', 'sub_start', 'sub_end_14d']]
 
 
train_sub = build_sub_dates(train['properties'])
test_sub  = build_sub_dates(test['properties'])
print(f'\n✅ Sub-date windows: Train {train_sub.shape}, Test {test_sub.shape}')

In [ ]:
# ==============================================================================
# CELL 3: Properties Features
# OOT-safe rules:
#   ✅ plan_tier, country_tier — stable categories
#   ✅ sub_start_is_night, sub_start_is_weekend — hour/weekday of signup (behavioral)
#   ❌ NO month, NO day-of-month, NO year — these encode time period → OOT leakage
#   ❌ NO sub_start_hour, NO sub_start_weekday as raw numbers
# ==============================================================================
 
PLAN_PRICE_MAP = {
    'Higgsfield Ultimate': 89.0,
    'Higgsfield Pro':      35.0,
    'Higgsfield Basic':     9.0,
    'Higgsfield Free':      1.0,
}
 
def build_prop_features(props_df: pd.DataFrame) -> pd.DataFrame:
    df = props_df.copy()
    sub_dt = parse_dates(df['subscription_start_date'])
 
    # Binary behavioral flags from signup time — OOT-safe (not month-based)
    df['sub_start_is_weekend'] = (sub_dt.dt.dayofweek >= 5).astype(int)
    df['sub_start_is_night']   = sub_dt.dt.hour.between(0, 6).astype(int)
 
    # Plan tier (ordinal encoding)
    plan_map = {
        'Higgsfield Ultimate': 3,
        'Higgsfield Pro':      2,
        'Higgsfield Basic':    1,
        'Higgsfield Free':     0,
    }
    df['plan_tier']       = df['subscription_plan'].map(plan_map).fillna(1).astype(int)
    df['plan_base_price'] = df['subscription_plan'].map(PLAN_PRICE_MAP).fillna(9.0)
 
    # Country tier (risk bucketing — stable across time)
    tier1 = {'US','GB','CA','AU','DE','FR','NL','SE','DK','NO','CH','JP','SG','NZ','FI','AT'}
    tier3 = {'BY','RU','UA','TR','NG','PK','KE','GH','VN','EG','PH','BD','MM','SD'}
    df['country_tier'] = df['country_code'].map(
        lambda x: 1 if x in tier1 else (3 if x in tier3 else 2)
    ).fillna(2).astype(int)
 
    # Country group — will be target-encoded in CV (with smoothing)
    top30 = props_df['country_code'].value_counts().nlargest(30).index
    df['country_group'] = (
        df['country_code'].where(df['country_code'].isin(top30), 'other')
        .fillna('unknown')
    )
 
    keep = [
        'user_id', 'plan_tier', 'plan_base_price',
        'sub_start_is_weekend', 'sub_start_is_night',
        'country_tier', 'country_group', 'subscription_plan',
    ]
    return df[keep]

In [ ]:
# ==============================================================================
# CELL 4: Quiz / Onboarding Features
# ==============================================================================
def build_quiz_features(quiz_df: pd.DataFrame) -> pd.DataFrame:
    # Some users appear twice — keep first row
    df = quiz_df.copy().drop_duplicates(subset='user_id', keep='first')
 
    cat_cols = ['source', 'flow_type', 'team_size', 'experience',
                'usage_plan', 'frustration', 'first_feature', 'role']
    for c in cat_cols:
        df[c] = df[c].fillna('unknown').astype(str)
 
    frustration_order = {
        'high-cost':   4,
        'limited':     3,
        'slow':        2,
        'hard-to-use': 2,
        'inconsistent':2,
        'other':       1,
        'unknown':     0,
    }
    exp_map = {'beginner': 1, 'intermediate': 2, 'advanced': 3, 'expert': 3, 'unknown': 0}
 
    df['frustration_score']    = df['frustration'].map(frustration_order).fillna(1)
    df['experience_level']     = df['experience'].map(exp_map).fillna(0)
    df['is_invited']           = (df['flow_type'] == 'invited').astype(int)
    df['is_team_user']         = (~df['team_size'].isin(['1', '1.0', 'unknown', ''])).astype(int)
    df['is_personal_use']      = (df['usage_plan'] == 'personal').astype(int)
    df['frustrated_highcost']  = (df['frustration'] == 'high-cost').astype(int)
    df['has_quiz']             = 1
 
    keep = [
        'user_id', 'source', 'usage_plan', 'first_feature', 'role',
        'frustration', 'frustration_score', 'experience_level',
        'is_invited', 'is_team_user', 'is_personal_use',
        'frustrated_highcost', 'has_quiz',
    ]
    return df[keep]

In [ ]:
# ==============================================================================
# CELL 5: Purchase Features
# OOT-safe rules:
#   ✅ Counts, binary flags — stable
#   ✅ days_since_sub (relative to user's own sub_start) — OOT-safe
#   ❌ Absolute dollar amounts removed — replaced by plan-relative ratios in Cell 8
#      (Prices could change between train/test periods)
# ==============================================================================
 
def build_purchase_features(purch_df: pd.DataFrame,
                             sub_dates: pd.DataFrame) -> pd.DataFrame:
    df = purch_df.copy()
    df['purchase_dt'] = parse_dates(df['purchase_time'])
    df = df.merge(sub_dates, on='user_id', how='left')
    df = df[
        (df['purchase_dt'] >= df['sub_start']) &
        (df['purchase_dt'] <= df['sub_end_14d'])
    ]
    df['days_since_sub'] = (
        (df['purchase_dt'] - df['sub_start']).dt.total_seconds() / 86400
    ).clip(lower=0)
 
    agg = df.groupby('user_id').agg(
        n_purchases         = ('purchase_type', 'count'),
        total_spend         = ('purchase_amount_dollars', 'sum'),   # normalized later
        n_credit_pkgs       = ('purchase_type', lambda x: (x == 'Credits package').sum()),
        n_sub_update        = ('purchase_type', lambda x: (x == 'Subscription Update').sum()),
        n_sub_create        = ('purchase_type', lambda x: (x == 'Subscription Create').sum()),
        last_purchase_day   = ('days_since_sub', 'max'),
        first_purchase_day  = ('days_since_sub', 'min'),
    ).reset_index()
 
    agg['bought_extra_credits'] = (agg['n_credit_pkgs'] > 0).astype(int)
    agg['upgraded_plan']        = (agg['n_sub_update']  > 0).astype(int)
    # Span of purchase activity within 14d window
    agg['purchase_span_days']   = (agg['last_purchase_day'] - agg['first_purchase_day']).clip(lower=0)
 
    return agg

In [ ]:
# ==============================================================================
# CELL 6: Transaction Features
# New features added:
#   ✅ threeds_friction   — bank required 3DS but user never authenticated
#   ✅ bank_country_mismatch — bank_country != card_country (geo risk)
#   ✅ all_attempts_failed — 100% failure rate (strong invol_churn signal)
#   ✅ bank_name — added to card_feats for target encoding later
# ==============================================================================
 
def build_txn_features(txn_df: pd.DataFrame,
                        sub_dates: pd.DataFrame) -> pd.DataFrame:
    df = txn_df.copy()
    df['txn_dt'] = parse_dates(df['transaction_time'])
    df = df.merge(sub_dates, on='user_id', how='left')
    df = df[
        (df['txn_dt'] >= df['sub_start']) &
        (df['txn_dt'] <= df['sub_end_14d'])
    ]
    df['days_since_sub'] = (
        (df['txn_dt'] - df['sub_start']).dt.total_seconds() / 86400
    ).clip(lower=0)
 
    # Failure flags
    df['is_failed']             = df['failure_code'].notna().astype(int)
    df['is_insufficient_funds'] = (df['failure_code'] == 'insufficient_funds').astype(int)
    df['is_do_not_honor']       = (df['failure_code'] == 'do_not_honor').astype(int)
    df['is_card_declined']      = (df['failure_code'] == 'card_declined').astype(int)
    df['cvc_failed']            = (df['cvc_check'] == 'fail').astype(int)
 
    # 3DS friction: bank requested 3DS but user did NOT authenticate
    df['is_3d_secure']               = df['is_3d_secure'].map({True: 1, False: 0, 'True': 1, 'False': 0}).fillna(0)
    df['is_3d_secure_authenticated'] = df['is_3d_secure_authenticated'].map({True: 1, False: 0, 'True': 1, 'False': 0}).fillna(0)
    df['threeds_friction'] = (
        (df['is_3d_secure'] == 1) & (df['is_3d_secure_authenticated'] == 0)
    ).astype(int)
 
    txn_agg = df.groupby('user_id').agg(
        n_txn_attempts        = ('transaction_id', 'count'),
        n_failed              = ('is_failed', 'sum'),
        n_insufficient_funds  = ('is_insufficient_funds', 'sum'),
        n_do_not_honor        = ('is_do_not_honor', 'sum'),
        n_card_declined       = ('is_card_declined', 'sum'),
        n_cvc_failed          = ('cvc_failed', 'sum'),
        n_threeds_friction    = ('threeds_friction', 'sum'),
        total_txn_amount      = ('amount_in_usd', 'sum'),   # normalized in master merge
        last_txn_day          = ('days_since_sub', 'max'),
        first_txn_day         = ('days_since_sub', 'min'),
    ).reset_index()
 
    denom = txn_agg['n_txn_attempts'].clip(lower=1)
    txn_agg['failure_rate']         = txn_agg['n_failed']             / denom
    txn_agg['cvc_fail_rate']        = txn_agg['n_cvc_failed']         / denom
    txn_agg['insuff_funds_rate']    = txn_agg['n_insufficient_funds'] / denom
    txn_agg['threeds_friction_rate'] = txn_agg['n_threeds_friction']  / denom
    txn_agg['all_attempts_failed']  = (txn_agg['failure_rate'] == 1.0).astype(int)
 
    # Top failure code per user (for target encoding)
    fail_mode = (
        df[df['failure_code'].notna()]
        .groupby('user_id')['failure_code']
        .agg(lambda x: x.value_counts().index[0] if len(x) > 0 else 'unknown')
        .reset_index()
        .rename(columns={'failure_code': 'top_failure_code'})
    )
 
    # Card-level features (last transaction = most recent card info)
    card_cols = [
        'card_brand', 'card_funding', 'digital_wallet',
        'card_3d_secure_support', 'card_country', 'billing_address_country',
        'bank_name', 'bank_country',
        'is_prepaid', 'is_virtual', 'is_business',
    ]
    card_feats = (
        df.sort_values('txn_dt')
        .groupby('user_id')[card_cols]
        .last()
        .reset_index()
    )
 
    # Boolean card flags
    for bool_col in ['is_prepaid', 'is_virtual', 'is_business']:
        if bool_col in card_feats.columns:
            card_feats[bool_col] = (
                card_feats[bool_col]
                .map({'True': 1, 'False': 0, True: 1, False: 0})
                .fillna(0).astype(int)
            )
 
    # Geo mismatch features
    card_feats['card_billing_mismatch'] = (
        card_feats['card_country'].fillna('x') !=
        card_feats['billing_address_country'].fillna('y')
    ).astype(int)
 
    card_feats['bank_card_country_mismatch'] = (
        card_feats['bank_country'].fillna('x') !=
        card_feats['card_country'].fillna('y')
    ).astype(int)
 
    return (
        txn_agg
        .merge(fail_mode,   on='user_id', how='left')
        .merge(card_feats,  on='user_id', how='left')
    )
 
 
print('Building properties, quiz, purchase, transaction features...')
feat_props_train = build_prop_features(train['properties'])
feat_props_test  = build_prop_features(test['properties'])
feat_quiz_train  = build_quiz_features(train['quizzes'])
feat_quiz_test   = build_quiz_features(test['quizzes'])
feat_purch_train = build_purchase_features(train['purchases'], train_sub)
feat_purch_test  = build_purchase_features(test['purchases'],  test_sub)
feat_txn_train   = build_txn_features(train['transactions'], train_sub)
feat_txn_test    = build_txn_features(test['transactions'],  test_sub)
print('✅ Done')

In [ ]:
# ==============================================================================
# CELL 7: Generation Features (Chunked — 5 GB file)
# New features:
#   ✅ time_to_first_success_hours  — delay from sub start to first success
#   ✅ failed_before_success        — had error BEFORE first success (frustration)
#   ✅ only_failures                — never succeeded at all
#   ✅ max_nsfw_per_day             — NSFW-seeking intensity per single day
#   ✅ gen_credit_burn_velocity     — credits used / active span
#   ✅ gen_dur_std                  — speed inconsistency (video only)
#   ✅ Proper weighted duration mean (sum/count across chunks)
#   ✅ Proper duration std (sum-of-squares method across chunks)
# ==============================================================================
 
def build_gen_features(gen_path: str, sub_dates: pd.DataFrame,
                        label: str, chunk_size: int = 3_000_000) -> pd.DataFrame:
    cache_path = Path(f'feat_gen_{label}.parquet')
    if cache_path.exists():
        print(f'🚀 Cache hit: {cache_path}')
        return pd.read_parquet(cache_path)
 
    USECOLS = ['user_id', 'status', 'credit_cost', 'generation_type',
               'created_at', 'duration']
 
    valid_users     = set(sub_dates['user_id'].values)
    IMAGE_TYPES     = {'image_model_1', 'image_model_2'}
    VIDEO_TYPES     = {'video_model_1', 'video_model_2'}
 
    agg_chunks       = []
    nsfw_day_chunks  = []   # for max_nsfw_per_day (needs cross-chunk dedup by day)
 
    reader = pd.read_csv(gen_path, usecols=USECOLS, chunksize=chunk_size, low_memory=False)
 
    for chunk in tqdm(reader, desc=f'Generations [{label}]'):
        # Filter to known users only — speeds up memory significantly
        chunk = chunk[chunk['user_id'].isin(valid_users)].copy()
        if chunk.empty:
            continue
 
        chunk['created_dt'] = parse_dates(chunk['created_at'])
        chunk = chunk.merge(
            sub_dates[['user_id', 'sub_start', 'sub_end_14d']],
            on='user_id', how='left'
        )
        # Keep only events within 14-day window
        chunk = chunk[
            (chunk['created_dt'] >= chunk['sub_start']) &
            (chunk['created_dt'] <= chunk['sub_end_14d'])
        ]
        if chunk.empty:
            continue
 
        chunk['days_since_sub'] = (
            (chunk['created_dt'] - chunk['sub_start']).dt.total_seconds() / 86400
        ).clip(lower=0)
        chunk['day_int'] = chunk['days_since_sub'].astype(int)
 
        chunk['is_completed'] = (chunk['status'] == 'completed').astype('int8')
        chunk['is_failed']    = chunk['status'].isin(['failed', 'error']).astype('int8')
        chunk['is_nsfw']      = (chunk['status'] == 'nsfw').astype('int8')
        chunk['is_video']     = chunk['generation_type'].isin(VIDEO_TYPES).astype('int8')
        chunk['is_image']     = chunk['generation_type'].isin(IMAGE_TYPES).astype('int8')
        chunk['is_week1']     = (chunk['days_since_sub'] <= 7).astype('int8')
        chunk['is_week2']     = (chunk['days_since_sub'] >  7).astype('int8')
 
        dur = chunk['duration'].fillna(np.nan)
 
        # Standard aggregations
        agg = chunk.groupby('user_id').agg(
            gen_total         = ('status', 'count'),
            gen_completed     = ('is_completed', 'sum'),
            gen_failed        = ('is_failed', 'sum'),
            gen_nsfw          = ('is_nsfw', 'sum'),
            gen_video         = ('is_video', 'sum'),
            gen_image         = ('is_image', 'sum'),
            gen_credit_sum    = ('credit_cost', 'sum'),
            gen_dur_sum       = ('duration', 'sum'),        # for weighted mean
            gen_dur_count     = ('duration', 'count'),      # non-NaN count
            gen_dur_sq_sum    = ('duration', lambda x: (x ** 2).sum()),  # for std
            gen_last_day      = ('days_since_sub', 'max'),
            gen_first_day     = ('days_since_sub', 'min'),
            gen_unique_types  = ('generation_type', 'nunique'),
            gen_unique_days   = ('day_int', 'nunique'),
            gen_week1         = ('is_week1', 'sum'),
            gen_week2         = ('is_week2', 'sum'),
        ).reset_index()
 
        # --- time_to_first_success & failed_before_success ---
        # Track min days_since_sub for completed and failed separately
        completed_rows = chunk[chunk['is_completed'] == 1][['user_id', 'days_since_sub']]
        failed_rows    = chunk[chunk['is_failed']    == 1][['user_id', 'days_since_sub']]
 
        if not completed_rows.empty:
            first_ok = (completed_rows.groupby('user_id')['days_since_sub']
                        .min().reset_index(name='first_success_day'))
            agg = agg.merge(first_ok, on='user_id', how='left')
        else:
            agg['first_success_day'] = np.nan
 
        if not failed_rows.empty:
            first_fail = (failed_rows.groupby('user_id')['days_since_sub']
                          .min().reset_index(name='first_failure_day'))
            agg = agg.merge(first_fail, on='user_id', how='left')
        else:
            agg['first_failure_day'] = np.nan
 
        # --- max_nsfw_per_day (need cross-chunk day dedup) ---
        nsfw_rows = chunk[chunk['is_nsfw'] == 1]
        if not nsfw_rows.empty:
            nsfw_day = (nsfw_rows.groupby(['user_id', 'day_int'])
                        .size().reset_index(name='nsfw_count'))
            nsfw_day_chunks.append(nsfw_day)
 
        agg_chunks.append(agg)
 
    if not agg_chunks:
        return pd.DataFrame(columns=['user_id'])
 
    combined = pd.concat(agg_chunks, ignore_index=True)
 
    # --- Final aggregation across chunks ---
    feat_gen = combined.groupby('user_id').agg(
        gen_total         = ('gen_total',         'sum'),
        gen_completed     = ('gen_completed',     'sum'),
        gen_failed        = ('gen_failed',        'sum'),
        gen_nsfw          = ('gen_nsfw',          'sum'),
        gen_video         = ('gen_video',         'sum'),
        gen_image         = ('gen_image',         'sum'),
        gen_credit_sum    = ('gen_credit_sum',    'sum'),
        gen_dur_sum       = ('gen_dur_sum',       'sum'),
        gen_dur_count     = ('gen_dur_count',     'sum'),
        gen_dur_sq_sum    = ('gen_dur_sq_sum',    'sum'),
        gen_last_day      = ('gen_last_day',      'max'),
        gen_first_day     = ('gen_first_day',     'min'),
        gen_unique_types  = ('gen_unique_types',  'max'),
        gen_unique_days   = ('gen_unique_days',   'max'),
        gen_week1         = ('gen_week1',         'sum'),
        gen_week2         = ('gen_week2',         'sum'),
        first_success_day = ('first_success_day', 'min'),
        first_failure_day = ('first_failure_day', 'min'),
    ).reset_index()
 
    # --- max_nsfw_per_day (cross-chunk day dedup) ---
    if nsfw_day_chunks:
        all_nsfw  = pd.concat(nsfw_day_chunks, ignore_index=True)
        nsfw_agg  = (all_nsfw.groupby(['user_id', 'day_int'])['nsfw_count']
                     .sum().reset_index())
        max_nsfw  = (nsfw_agg.groupby('user_id')['nsfw_count']
                     .max().reset_index(name='max_nsfw_per_day'))
        feat_gen  = feat_gen.merge(max_nsfw, on='user_id', how='left')
    else:
        feat_gen['max_nsfw_per_day'] = 0
 
    feat_gen['max_nsfw_per_day'] = feat_gen['max_nsfw_per_day'].fillna(0)
 
    # --- Rate features ---
    total_c = feat_gen['gen_total'].clip(lower=1)
    feat_gen['gen_complete_rate'] = feat_gen['gen_completed'] / total_c
    feat_gen['gen_fail_rate']     = feat_gen['gen_failed']    / total_c
    feat_gen['gen_nsfw_rate']     = feat_gen['gen_nsfw']      / total_c
    feat_gen['gen_video_ratio']   = feat_gen['gen_video']     / total_c
 
    # --- Proper duration mean (weighted across chunks) ---
    feat_gen['gen_dur_mean'] = np.where(
        feat_gen['gen_dur_count'] > 0,
        feat_gen['gen_dur_sum'] / feat_gen['gen_dur_count'].clip(lower=1),
        np.nan,
    )
 
    # --- Duration std via sum-of-squares (numerically stable across chunks) ---
    # Var = E[x^2] - E[x]^2,  Std = sqrt(max(Var, 0))
    feat_gen['gen_dur_std'] = np.where(
        feat_gen['gen_dur_count'] > 1,
        np.sqrt(np.maximum(
            feat_gen['gen_dur_sq_sum'] / feat_gen['gen_dur_count'].clip(lower=1) -
            feat_gen['gen_dur_mean'] ** 2,
            0,
        )),
        np.nan,
    )
 
    # --- Time-to-first-success (hours) ---
    # Users who never had a success get 14*24 = 336h (full window = max penalty)
    feat_gen['time_to_first_success_hours'] = (
        feat_gen['first_success_day']
        .fillna(14.0) * 24
    )
    feat_gen['has_any_success'] = feat_gen['first_success_day'].notna().astype(int)
 
    # --- Failed before success ---
    feat_gen['failed_before_success'] = (
        feat_gen['first_failure_day'].notna() &
        feat_gen['first_success_day'].notna() &
        (feat_gen['first_failure_day'] < feat_gen['first_success_day'])
    ).astype(int)
 
    # --- Only failures, never succeeded ---
    feat_gen['only_failures'] = (
        (feat_gen['gen_failed'] > 0) &
        (feat_gen['gen_completed'] == 0)
    ).astype(int)
 
    # --- Engagement features ---
    days_c = feat_gen['gen_unique_days'].clip(lower=1)
    feat_gen['gen_per_active_day']    = (feat_gen['gen_total'] / days_c).clip(upper=300)
    feat_gen['gen_active_span']       = (feat_gen['gen_last_day'] - feat_gen['gen_first_day']).clip(lower=0)
    feat_gen['gen_engagement_drop']   = feat_gen['gen_week1'] - feat_gen['gen_week2']
 
    # Credit burn velocity (only meaningful where credit_cost is recorded)
    feat_gen['gen_credit_burn_vel'] = (
        feat_gen['gen_credit_sum'] /
        (feat_gen['gen_active_span'] + 1).clip(lower=0.1)
    )
    feat_gen['has_credit_data'] = (feat_gen['gen_credit_sum'] > 0).astype(int)
 
    # Drop intermediate aggregation columns — not needed as features
    drop_cols = [
        'gen_dur_sum', 'gen_dur_count', 'gen_dur_sq_sum',
        'first_success_day', 'first_failure_day',
    ]
    feat_gen.drop(columns=drop_cols, errors='ignore', inplace=True)
 
    feat_gen.to_parquet(cache_path)
    print(f'✅ Saved: {cache_path}  shape={feat_gen.shape}')
    del combined, agg_chunks, nsfw_day_chunks
    gc.collect()
    return feat_gen
 
 
feat_gen_train = build_gen_features(GEN_TRAIN, train_sub, label='train')
feat_gen_test  = build_gen_features(GEN_TEST,  test_sub,  label='test')

In [ ]:
# ==============================================================================
# CELL 8: Master Merge + Spend Normalization + Interaction Features
# Key rules:
#   ✅ All absolute spend amounts normalized by plan_base_price
#   ✅ total_spend (absolute) DROPPED after ratio is computed
#   ✅ total_txn_amount (absolute) DROPPED after ratio is computed
#   ✅ No date-absolute features propagated
# ==============================================================================
 
def build_master(users_df: pd.DataFrame,
                 feat_props:  pd.DataFrame,
                 feat_quiz:   pd.DataFrame,
                 feat_purch:  pd.DataFrame,
                 feat_txn:    pd.DataFrame,
                 feat_gen:    pd.DataFrame,
                 include_target: bool = True) -> pd.DataFrame:
 
    cols = ['user_id', 'churn_status'] if include_target else ['user_id']
    df = users_df[cols].copy()
 
    df = (df
          .merge(feat_props, on='user_id', how='left')
          .merge(feat_quiz,  on='user_id', how='left')
          .merge(feat_purch, on='user_id', how='left')
          .merge(feat_txn,   on='user_id', how='left')
          .merge(feat_gen,   on='user_id', how='left'))
 
    # ── Presence flags ────────────────────────────────────────────────────────
    df['has_purchases']    = df['n_purchases'].notna().astype(int)
    df['has_transactions'] = df['n_txn_attempts'].notna().astype(int)
    df['has_generations']  = df['gen_total'].notna().astype(int)
    df['has_quiz']         = df['has_quiz'].notna().astype(int) \
                             if 'has_quiz' in df.columns else 0
 
    # ── Spend normalization (OOT protection) ──────────────────────────────────
    plan_price = df['plan_base_price'].fillna(9.0).clip(lower=1.0)
 
    # Purchases: extra spend beyond subscription price
    if 'total_spend' in df.columns:
        df['extra_spend_ratio']   = (df['total_spend'] - plan_price) / plan_price
        df['credits_to_plan_ratio'] = (
            df['n_credit_pkgs'].fillna(0) * 5.0 / plan_price  # ~$5/pkg
        )
        df.drop(columns=['total_spend'], inplace=True, errors='ignore')
 
    # Transactions: total attempted amount relative to plan
    if 'total_txn_amount' in df.columns:
        df['txn_amount_to_plan_ratio'] = df['total_txn_amount'] / plan_price
        df.drop(columns=['total_txn_amount'], inplace=True, errors='ignore')
 
    # Drop plan_base_price (it's encoded in plan_tier/subscription_plan already)
    df.drop(columns=['plan_base_price'], inplace=True, errors='ignore')
 
    # ── Recency feature ────────────────────────────────────────────────────────
    activity_cols = [c for c in ['gen_last_day', 'last_purchase_day', 'last_txn_day']
                     if c in df.columns]
    if activity_cols:
        df['max_active_day']        = df[activity_cols].max(axis=1)
        df['days_silent_at_end']    = (14 - df['max_active_day'].fillna(0)).clip(lower=0)
 
    # ── Risk interaction features ──────────────────────────────────────────────
    df['prepaid_high_risk_country'] = (
        (df.get('is_prepaid', 0).fillna(0) == 1) &
        (df['country_tier'] == 3)
    ).astype(int)
 
    df['payment_and_gen_issues'] = (
        (df.get('failure_rate', pd.Series(0, index=df.index)).fillna(0) > 0.3) &
        (df.get('gen_fail_rate', pd.Series(0, index=df.index)).fillna(0) > 0.2)
    ).astype(int)
 
    df['zero_spend_user'] = (df['has_purchases'] == 0).astype(int)
 
    # High frustration + no successful generation = strong vol_churn signal
    df['frustrated_no_success'] = (
        (df.get('frustration_score', pd.Series(0, index=df.index)).fillna(0) >= 3) &
        (df.get('has_any_success', pd.Series(1, index=df.index)).fillna(1) == 0)
    ).astype(int)
 
    # Slow first success + high cost frustration
    df['slow_start_highcost'] = (
        (df.get('time_to_first_success_hours', pd.Series(0, index=df.index)).fillna(0) > 48) &
        (df.get('frustrated_highcost', pd.Series(0, index=df.index)).fillna(0) == 1)
    ).astype(int)
 
    # Virtual/prepaid card — strong invol_churn indicator
    df['high_risk_card'] = (
        df.get('is_prepaid',  pd.Series(0, index=df.index)).fillna(0).astype(int) |
        df.get('is_virtual',  pd.Series(0, index=df.index)).fillna(0).astype(int)
    ).astype(int)
 
    # Clip outlier gen_per_active_day
    if 'gen_per_active_day' in df.columns:
        df['gen_per_active_day'] = df['gen_per_active_day'].clip(upper=300)
 
    return df
 
 
master_train = build_master(
    train['users'], feat_props_train, feat_quiz_train,
    feat_purch_train, feat_txn_train, feat_gen_train, True
)
master_test = build_master(
    test['users'], feat_props_test, feat_quiz_test,
    feat_purch_test, feat_txn_test, feat_gen_test, False
)
print(f'Master train: {master_train.shape}')
print(f'Master test:  {master_test.shape}')

In [ ]:
# ==============================================================================
# CELL 9: Preprocessing — Label Encoding & Feature Setup
#
# Column strategy:
#   LABEL_ENC_COLS  → scikit LabelEncoder (safe cardinality)
#   TARGET_ENC_COLS → ce.TargetEncoder inside CV folds (high cardinality / OOT)
#   DROP_COLS       → not features
#
# payment_method_type dropped — always 'card' (zero variance)
# bank_country dropped — redundant with bank_card_country_mismatch
# ==============================================================================
 
LABEL_ENC_COLS = [
    'subscription_plan', 'source', 'usage_plan', 'first_feature',
    'role', 'frustration',
    'card_brand', 'card_funding', 'digital_wallet', 'card_3d_secure_support',
    'card_country', 'billing_address_country',
]
 
# High-cardinality cols that benefit from smoothed target encoding
TARGET_ENC_COLS = ['country_group', 'top_failure_code', 'bank_name']
 
DROP_COLS = [
    'user_id', 'churn_status',
    'payment_method_type',   # always 'card'
    'bank_country',          # encoded via bank_card_country_mismatch
]
 
# ── Target label encoding ──────────────────────────────────────────────────────
le_target = LabelEncoder()
le_target.fit(['invol_churn', 'not_churned', 'vol_churn'])
print('Target classes:', le_target.classes_)
 
y_all = le_target.transform(master_train['churn_status'])
print('Class distribution:', dict(zip(le_target.classes_,
      np.bincount(y_all))))
 
 
# ── Label encoding (global — safe, no target leakage) ─────────────────────────
def apply_label_encoding(df_train: pd.DataFrame, df_test: pd.DataFrame,
                          cols: list) -> tuple:
    tr, te = df_train.copy(), df_test.copy()
    for col in cols:
        if col not in tr.columns:
            continue
        tr[col] = tr[col].fillna('unknown').astype(str)
        le = LabelEncoder()
        le.fit(list(tr[col].unique()) + ['unknown'])
        tr[col] = le.transform(tr[col])
        if col in te.columns:
            te[col] = te[col].fillna('unknown').astype(str)
            te[col] = te[col].apply(lambda v: v if v in le.classes_ else 'unknown')
            te[col] = le.transform(te[col])
        else:
            te[col] = le.transform(['unknown'])[0]
    return tr, te
 
 
master_train_enc, master_test_enc = apply_label_encoding(
    master_train, master_test, LABEL_ENC_COLS
)
 
# ── Build X_train / X_test ────────────────────────────────────────────────────
cols_to_drop_in_X = [c for c in DROP_COLS if c in master_train_enc.columns]
feat_cols = [c for c in master_train_enc.columns if c not in cols_to_drop_in_X]
 
X_train = master_train_enc[feat_cols].copy()
X_test  = master_test_enc[[c for c in feat_cols if c in master_test_enc.columns]].copy()
 
# Ensure test has all columns in same order
for c in feat_cols:
    if c not in X_test.columns:
        X_test[c] = 'unknown' if c in TARGET_ENC_COLS else 0
 
X_test = X_test[feat_cols]
 
# Fill NaNs — string 'unknown' for TE cols, 0 for numeric
for col in feat_cols:
    if col in TARGET_ENC_COLS:
        X_train[col] = X_train[col].fillna('unknown').astype(str)
        X_test[col]  = X_test[col].fillna('unknown').astype(str)
    else:
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce').fillna(0)
        X_test[col]  = pd.to_numeric(X_test[col],  errors='coerce').fillna(0)
 
print(f'\nX_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'Features: {len(feat_cols)}')

In [ ]:
# ==============================================================================
# CELL 10: Leak-Free Cross-Validation (LGBM + XGBoost) - NO BALANCING
# ==============================================================================
import xgboost as xgb

# Параметры БЕЗ class_weight='balanced'
LGBM_PARAMS = dict(
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=127,
    max_depth=8,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    importance_type='gain',
    verbose=-1,
)

XGB_PARAMS = dict(
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=7,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    n_jobs=-1,
    tree_method='hist', # 'gpu_hist' если есть GPU
    objective='multi:softprob',
    eval_metric='mlogloss',
)

LGBM_CAT_COLS = [c for c in LABEL_ENC_COLS + ['country_tier'] if c in X_train.columns]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_preds = np.zeros((len(X_train), 3))
best_iters_lgb = []
best_iters_xgb = []

print('=' * 60)
print('  STARTING ENSEMBLE CV (LGBM + XGB)')
print(f'  Target classes: {le_target.classes_}')
print('=' * 60)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_all)):
    X_tr,  X_val  = X_train.iloc[tr_idx].copy(), X_train.iloc[val_idx].copy()
    y_tr,  y_val  = y_all[tr_idx],               y_all[val_idx]

    # Target encoding внутри фолда
    te = ce.TargetEncoder(cols=TARGET_ENC_COLS, smoothing=30.0)
    X_tr[TARGET_ENC_COLS]  = te.fit_transform(X_tr[TARGET_ENC_COLS], y_tr)
    X_val[TARGET_ENC_COLS] = te.transform(X_val[TARGET_ENC_COLS])

    # 1. LightGBM с логами
    model_lgb = lgb.LGBMClassifier(**LGBM_PARAMS)
    model_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        categorical_feature=LGBM_CAT_COLS,
        callbacks=[
            lgb.early_stopping(150),
            lgb.log_evaluation(period=100) # логи каждые 100 итераций
        ]
    )
    p_lgb = model_lgb.predict_proba(X_val)
    best_iters_lgb.append(model_lgb.best_iteration_)

    # 2. XGBoost с логами
    model_xgb = xgb.XGBClassifier(**XGB_PARAMS, early_stopping_rounds=150)
    model_xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=100 # логи каждые 100 итераций
    )
    p_xgb = model_xgb.predict_proba(X_val)
    best_iters_xgb.append(model_xgb.best_iteration)

    # Усреднение вероятностей (Simple Blend)
    val_probs = (p_lgb + p_xgb) / 2.0
    oof_preds[val_idx] = val_probs

    fold_f1 = f1_score(y_val, val_probs.argmax(axis=1), average='weighted')
    print(f'\n>>> Fold {fold+1} Finished | Weighted F1: {fold_f1:.4f}')
    print('-' * 40)

# Финальный отчет по OOF
oof_labels = oof_preds.argmax(axis=1)
oof_wf1 = f1_score(y_all, oof_labels, average='weighted')
print(f'\n[FINAL OOF ENSEMBLE RESULTS]')
print(f'Weighted F1: {oof_wf1:.4f}')
print(classification_report(y_all, oof_labels, target_names=le_target.classes_))

In [ ]:
# ==============================================================================
# CELL 11: Final Full Train & Submission
# ==============================================================================

# Среднее количество итераций + 10% запас
final_n_lgb = int(np.mean(best_iters_lgb) * 1.1)
final_n_xgb = int(np.mean(best_iters_xgb) * 1.1)

print(f'Training final models: LGBM ({final_n_lgb} iters), XGB ({final_n_xgb} iters)...')

# Подготовка финальных данных
te_final = ce.TargetEncoder(cols=TARGET_ENC_COLS, smoothing=30.0)
X_train_final = X_train.copy()
X_test_final  = X_test.copy()

X_train_final[TARGET_ENC_COLS] = te_final.fit_transform(X_train_final[TARGET_ENC_COLS], y_all)
X_test_final[TARGET_ENC_COLS]  = te_final.transform(X_test_final[TARGET_ENC_COLS])

# Чистим типы данных
for col in feat_cols:
    if col not in TARGET_ENC_COLS:
        X_train_final[col] = pd.to_numeric(X_train_final[col], errors='coerce').fillna(0)
        X_test_final[col]  = pd.to_numeric(X_test_final[col],  errors='coerce').fillna(0)

# Финальный LightGBM
final_lgb = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'n_estimators': final_n_lgb})
final_lgb.fit(X_train_final, y_all, categorical_feature=LGBM_CAT_COLS)

# Финальный XGBoost
xgb_final_params = XGB_PARAMS.copy()
xgb_final_params.pop('eval_metric', None) # Чистим для фита без eval_set
final_xgb = xgb.XGBClassifier(**{**xgb_final_params, 'n_estimators': final_n_xgb})
final_xgb.fit(X_train_final, y_all)

# Предсказания на тесте
p_test_lgb = final_lgb.predict_proba(X_test_final)
p_test_xgb = final_xgb.predict_proba(X_test_final)

# Ансамбль (Soft Voting)
test_probs = (p_test_lgb + p_test_xgb) / 2.0
test_preds = test_probs.argmax(axis=1)
final_labels = le_target.inverse_transform(test_preds)

# Распределение предиктов (проверка на адекватность)
print('\n[Test Class Distribution]')
print(pd.Series(final_labels).value_counts())

# Сохранение
submission = pd.DataFrame({
    'user_id': test['users']['user_id'].values,
    'churn_status': final_labels
})

submission.to_csv('submission.csv', index=False)
print('\n✅ submission.csv created! Good luck at HackNU!')

In [ ]:
submission

In [ ]:
import joblib

# 1. Сохраняем модели
joblib.dump(final_lgb, 'lgbm_churn_model.pkl')
joblib.dump(final_xgb, 'xgb_churn_model.pkl')

# 2. Сохраняем энкодеры (БЕЗ НИХ МОДЕЛЬ БУДЕТ БЕСПОЛЕЗНА)
joblib.dump(le_target, 'target_label_encoder.pkl')
joblib.dump(te_final, 'target_high_card_encoder.pkl')

# Сохраняем список фичей, чтобы не запутаться в порядке колонок
joblib.dump(feat_cols, 'feature_columns.pkl')

print("✅ Все артефакты сохранены в /kaggle/working/")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Извлекаем важность из LightGBM (у нас тип 'gain' — это вклад в точность)
lgb_importance = pd.DataFrame({
    'feature': feat_cols,
    'importance_lgb': final_lgb.feature_importances_
})

# 2. Извлекаем важность из XGBoost (у него по умолчанию 'gain')
xgb_importance = pd.DataFrame({
    'feature': feat_cols,
    'importance_xgb': final_xgb.feature_importances_
})

# 3. Нормализуем значения (чтобы шкалы моделей были сопоставимы) и усредняем
fi = pd.merge(lgb_importance, xgb_importance, on='feature')
fi['importance_lgb'] /= fi['importance_lgb'].sum()
fi['importance_xgb'] /= fi['importance_xgb'].sum()
fi['combined_importance'] = (fi['importance_lgb'] + fi['importance_xgb']) / 2

# 4. Сортируем и берем ТОП-10
top_10_features = fi.sort_values('combined_importance', ascending=False).head(10)

print("=== ТОП-10 САМЫХ ВАЖНЫХ ФИЧЕЙ ДЛЯ БИЗНЕСА ===")
for i, row in enumerate(top_10_features.itertuples(), 1):
    print(f"{i}. {row.feature: <30} | Относительный вес: {row.combined_importance:.4f}")

# Визуализация
plt.figure(figsize=(12, 8))
sns.barplot(data=top_10_features, x='combined_importance', y='feature', palette='viridis')
plt.title('Top 10 Features by Combined Gain (LGBM + XGB)')
plt.xlabel('Normalized Importance')
plt.ylabel('Feature Name')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
def deep_bank_analysis(df, min_users=50):
    # Готовим данные
    df['is_churn'] = (df['churn_status'] != 'not_churned').astype(int)
    
    bank_stats = df.groupby('bank_name').agg({
        'user_id': 'count',
        'is_churn': 'mean'
    }).rename(columns={'user_id': 'user_count', 'is_churn': 'churn_rate'})
    
    # Фильтруем редкие банки, чтобы не было "шума"
    stats_filtered = bank_stats[bank_stats['user_count'] >= min_users].copy()
    
    plt.figure(figsize=(14, 7))
    sns.scatterplot(data=stats_filtered, x='user_count', y='churn_rate', 
                    size='user_count', sizes=(50, 500), alpha=0.6, hue='churn_rate', palette='coolwarm')
    
    # Добавляем названия для самых экстремальных банков
    top_risk = stats_filtered.sort_values('churn_rate', ascending=False).head(5)
    for i in range(len(top_risk)):
        plt.text(top_risk.user_count.iloc[i]+5, top_risk.churn_rate.iloc[i], 
                 top_risk.index[i], fontsize=9, weight='bold')

    plt.axhline(df['is_churn'].mean(), color='black', linestyle='--', label='Avg Churn Rate')
    plt.title(f'Bank Reliability Map (Banks with >{min_users} users)')
    plt.xlabel('Number of Users (Volume)')
    plt.ylabel('Churn Rate (Risk)')
    plt.grid(True, alpha=0.2)
    plt.show()

deep_bank_analysis(master_train, min_users=30)

In [ ]:
def plot_risk_matrix_fixed(df):
    temp_df = df.copy()
    temp_df['is_churn'] = (temp_df['churn_status'] != 'not_churned').astype(int)
    temp_df['is_mismatch'] = temp_df['card_billing_mismatch'].fillna(0).map({1: 'Mismatch', 0: 'Match'})
    temp_df['card_type'] = temp_df['high_risk_card'].fillna(0).map({1: 'Virtual/Prepaid', 0: 'Standard'})
    
    # Исправляем ошибку: используем aggfunc
    pivot = temp_df.pivot_table(
        index='card_type', 
        columns='is_mismatch', 
        values='is_churn', 
        aggfunc='mean'
    )
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(pivot, annot=True, fmt=".1%", cmap='YlOrRd', annot_kws={"size": 14})
    plt.title('Churn Rate: The "Fraud/Trial Abuse" Matrix')
    plt.show()

plot_risk_matrix_fixed(master_train)

In [ ]:
def analyze_aha_moment_fixed(df):
    # Создаем копию и добавляем таргет прямо в неё
    active_users = df[df['has_generations'] == 1].copy()
    active_users['is_churn'] = (active_users['churn_status'] != 'not_churned').astype(int)
    
    # Создаем временные интервалы
    active_users['time_to_success_bin'] = pd.cut(
        active_users['time_to_first_success_hours'], 
        bins=[0, 1, 6, 12, 24, 48, 72, 336],
        labels=['<1h', '1-6h', '6-12h', '12-24h', '1-2d', '2-3d', 'Long Time']
    )
    
    plt.figure(figsize=(12, 6))
    # Теперь и x, и y из одного DataFrame — ошибки не будет
    sns.barplot(data=active_users, x='time_to_success_bin', y='is_churn', palette='viridis')
    
    plt.axhline(active_users['is_churn'].mean(), color='red', linestyle='--', label='Avg Churn')
    plt.title('Impact of "Time to First Success" on Churn')
    plt.ylabel('Churn Probability (%)')
    plt.xlabel('Hours from Signup to First Success')
    plt.legend()
    plt.show()

analyze_aha_moment_fixed(master_train)

In [ ]:
import pandas as pd
import numpy as np

# 1. Подготовка данных: восстанавливаем текстовые метки классов
df_ana = master_train.copy()
# y_all — это закодированные метки (0, 1, 2). Возвращаем их в текст:
# Классы в le_target: ['invol_churn', 'not_churned', 'vol_churn']
df_ana['target_text'] = le_target.inverse_transform(y_all)

# Считаем глобальные средние для сравнения
AVG_VOL = (df_ana['target_text'] == 'vol_churn').mean()
AVG_INVOL = (df_ana['target_text'] == 'invol_churn').mean()

def deep_churn_report(df, feature, feature_name, is_continuous=False, n_min=30):
    print(f"\n" + "="*90)
    print(f"БИЗНЕС-АНАЛИЗ: {feature_name} ({feature}) | Порог N >= {n_min}")
    print("="*90)
    
    temp = df.copy()
    if is_continuous:
        temp[feature] = pd.qcut(temp[feature].rank(method='first'), q=5, 
                                labels=['1. Very Low', '2. Low', '3. Medium', '4. High', '5. Extreme'])
    
    # Считаем статистики по группам
    stats = temp.groupby(feature).agg(
        total_users=('user_id', 'count'),
        vol_rate=('target_text', lambda x: (x == 'vol_churn').mean()),
        invol_rate=('target_text', lambda x: (x == 'invol_churn').mean())
    ).reset_index()
    
    # Фильтруем редкие группы
    stats = stats[stats['total_users'] >= n_min]
    if stats.empty:
        print(f"⚠ Группы с N >= {n_min} не найдены.")
        return

    # Сортируем по суммарному оттоку
    stats['total_churn'] = stats['vol_rate'] + stats['invol_rate']
    stats = stats.sort_values('total_churn', ascending=False)

    # Печать таблицы
    header = f"{'Группа':<30} | {'Users':<7} | {'Vol %':<8} | {'vs Avg':<8} | {'Invol %':<8} | {'vs Avg'}"
    print(header)
    print("-" * len(header))
    
    for _, row in stats.iterrows():
        v_diff = row['vol_rate'] - AVG_VOL
        i_diff = row['invol_rate'] - AVG_INVOL
        
        v_str = f"{v_diff:+.1%}" if abs(v_diff) > 0.01 else "OK"
        i_str = f"{i_diff:+.1%}" if abs(i_diff) > 0.01 else "OK"
        
        val = str(row[feature])[:30]
        print(f"{val:<30} | {int(row['total_users']):<7} | "
              f"{row['vol_rate']:<8.1%} | {v_str:<8} | "
              f"{row['invol_rate']:<8.1%} | {i_str}")

# --- ЗАПУСК ПО ТОП-10 ФИЧАМ ---

features_to_analyze = [
    ('bank_name', "Банк Эмитент", False),
    ('card_3d_secure_support', "Поддержка 3DS", False),
    ('card_country', "Страна Карты", False),
    ('billing_address_country', "Страна Биллинга", False),
    ('card_billing_mismatch', "Гео-несовпадение", False),
    ('gen_credit_burn_vel', "Скорость траты кредитов", True),
    ('country_group', "Группы стран", False),
    ('is_business', "Тип карты (Business)", False),
    ('high_risk_card', "Виртуальная карта", False),
    ('gen_per_active_day', "Интенсивность использования", True)
]

for feat, name, cont in features_to_analyze:
    deep_churn_report(df_ana, feat, name, is_continuous=cont, n_min=30)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Подготовка данных (если еще не сделана)
df_ana = master_train.copy()
df_ana['target_text'] = le_target.inverse_transform(y_all)

# Глобальные константы
AVG_VOL = (df_ana['target_text'] == 'vol_churn').mean()
AVG_INVOL = (df_ana['target_text'] == 'invol_churn').mean()

def plot_deep_churn_visuals(df, feature, feature_name, is_continuous=False, n_min=30, top_n=15):
    temp = df.copy()
    
    # Обработка числовых признаков
    if is_continuous:
        temp[feature] = pd.qcut(temp[feature].rank(method='first'), q=5, 
                                labels=['1. Very Low', '2. Low', '3. Medium', '4. High', '5. Extreme'])
    
    # Агрегация статистики
    stats = temp.groupby(feature).agg(
        total_users=('user_id', 'count'),
        vol_rate=('target_text', lambda x: (x == 'vol_churn').mean()),
        invol_rate=('target_text', lambda x: (x == 'invol_churn').mean())
    ).reset_index()
    
    # Фильтрация по N >= 30
    stats = stats[stats['total_users'] >= n_min]
    if stats.empty:
        return

    # Сортировка по суммарному риску и выборка Топ-N
    stats['total_churn'] = stats['vol_rate'] + stats['invol_rate']
    stats = stats.sort_values('total_churn', ascending=False).head(top_n)

    # Подготовка данных для Seaborn (Melt)
    plot_data = stats.melt(id_vars=[feature, 'total_users'], 
                           value_vars=['vol_rate', 'invol_rate'], 
                           var_name='Churn Type', value_name='Rate')
    
    # Настройка размера графика в зависимости от кол-ва категорий
    plt.figure(figsize=(12, max(6, len(stats) * 0.6)))
    
    # Основной график
    palette = {'vol_rate': '#FF8C00', 'invol_rate': '#DC143C'} # Оранжевый и Красный
    ax = sns.barplot(data=plot_data, y=feature, x='Rate', hue='Churn Type', palette=palette)
    
    # Добавляем линии среднего значения
    plt.axvline(AVG_VOL, color='#FF8C00', linestyle='--', alpha=0.6, label=f'Global Vol Avg ({AVG_VOL:.1%})')
    plt.axvline(AVG_INVOL, color='#DC143C', linestyle='--', alpha=0.6, label=f'Global Invol Avg ({AVG_INVOL:.1%})')
    
    # Аннотации с количеством юзеров (N=...)
    for i, val in enumerate(stats['total_users']):
        plt.text(0.005, i, f"N={int(val)}", color='white', va='center', fontweight='bold', fontsize=9)

    # Оформление
    plt.title(f'Deep Dive: {feature_name}\n(Voluntary vs Involuntary Churn | Min N={n_min})', fontsize=15)
    plt.xlabel('Churn Probability (%)', fontsize=12)
    plt.ylabel('')
    plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(axis='x', alpha=0.2)
    plt.tight_layout()
    
    # Сохранение файла
    filename = f'churn_analysis_{feature}.png'
    plt.savefig(filename)
    print(f"✅ График сохранен: {filename}")
    plt.show()

# --- ЗАПУСК ВИЗУАЛИЗАЦИИ ДЛЯ ТОП-10 ---

features_to_plot = [
    ('bank_name', "Bank Name (Top 15 Riskiest)", False),
    ('card_3d_secure_support', "3D Secure Support", False),
    ('card_country', "Card Issuing Country", False),
    ('billing_address_country', "Billing Address Country", False),
    ('card_billing_mismatch', "Geo Mismatch (Card != Billing)", False),
    ('gen_credit_burn_vel', "Credit Burn Velocity (Quintiles)", True),
    ('country_group', "Regional Groups", False),
    ('is_business', "Card Type: Business", False),
    ('high_risk_card', "Card Type: Virtual/Prepaid", False),
    ('gen_per_active_day', "Usage Intensity (Quintiles)", True)
]

for feat, name, cont in features_to_plot:
    plot_deep_churn_visuals(df_ana, feat, name, is_continuous=cont, n_min=30)

In [ ]:
# ==============================================================================
# CELL 13: Explainability Engine — explain_user(user_id)
# ==============================================================================
import pandas as pd
import numpy as np


def explain_user(
    user_id,
    master_train_df,
    master_test_df,
    model_lgb,
    model_xgb,
    X_tr_final,
    X_te_final,
    label_encoder,
    min_bank_users: int = 30,
    deviation_threshold: float = 0.05,
    show_all_features: bool = False,
):
    """
    Explainability engine for churn prediction.

    Parameters
    ----------
    user_id              : any         — user to explain
    master_train_df      : DataFrame   — master_train  (raw, pre-encoding)
    master_test_df       : DataFrame   — master_test   (raw, pre-encoding)
    model_lgb            : fitted      — final_lgb
    model_xgb            : fitted      — final_xgb
    X_tr_final           : DataFrame   — X_train_final (encoded, for predict)
    X_te_final           : DataFrame   — X_test_final  (encoded, for predict)
    label_encoder        : LabelEncoder — le_target
    min_bank_users       : int         — min N to trust a bank's churn rate
    deviation_threshold  : float       — min abs Δ to flag a risk factor (e.g. 0.05 = 5pp)
    show_all_features    : bool        — also print protective / neutral factors
    """

    # ── 1. Locate user ────────────────────────────────────────────────────────
    in_train = user_id in master_train_df["user_id"].values
    in_test  = user_id in master_test_df["user_id"].values

    if not in_train and not in_test:
        print(f"❌  User {user_id} not found in train or test.")
        return None

    src_df = master_train_df if in_train else master_test_df
    src_X  = X_tr_final      if in_train else X_te_final

    # Positional lookup — safe regardless of DataFrame index
    uid_list = src_df["user_id"].tolist()
    if user_id not in uid_list:
        print(f"❌  user_id {user_id} not found in the chosen split.")
        return None
    pos = uid_list.index(user_id)

    user_raw = src_df.iloc[pos]
    user_X   = src_X.iloc[[pos]]
    true_label = user_raw.get("churn_status") if in_train else None

    # ── 2. Predict ────────────────────────────────────────────────────────────
    p_lgb  = model_lgb.predict_proba(user_X)[0]
    p_xgb  = model_xgb.predict_proba(user_X)[0]
    probs  = (p_lgb + p_xgb) / 2.0

    pred_idx   = probs.argmax()
    pred_label = label_encoder.inverse_transform([pred_idx])[0]
    classes    = label_encoder.classes_          # ['invol_churn','not_churned','vol_churn']
    prob_dict  = {c: probs[i] for i, c in enumerate(classes)}

    # ── 3. Header ─────────────────────────────────────────────────────────────
    W = 74
    SEP  = "═" * W
    sep  = "─" * W
    ICONS = {"invol_churn": "🔴", "vol_churn": "🟠", "not_churned": "🟢"}
    print(SEP)
    print(f"  EXPLAINABILITY REPORT  |  user_id = {user_id}")
    print(SEP)

    icon = ICONS[pred_label]
    print(f"\n{icon}  PREDICTION : {pred_label.upper()}")
    print(f"\n   {'Model':<12} {'invol_churn':>12}  {'not_churned':>12}  {'vol_churn':>12}")
    print(f"   {sep[:52]}")
    print(f"   {'LightGBM':<12} {p_lgb[0]:>12.1%}  {p_lgb[1]:>12.1%}  {p_lgb[2]:>12.1%}")
    print(f"   {'XGBoost':<12} {p_xgb[0]:>12.1%}  {p_xgb[1]:>12.1%}  {p_xgb[2]:>12.1%}")
    print(f"   {'Ensemble':<12} {prob_dict['invol_churn']:>12.1%}  {prob_dict['not_churned']:>12.1%}  {prob_dict['vol_churn']:>12.1%}")

    if true_label:
        match = "✅ correct" if pred_label == true_label else f"❌ actual = {true_label}"
        print(f"\n   Ground truth  :  {match}")

    # ── 4. Global averages ────────────────────────────────────────────────────
    avg_vol   = (master_train_df["churn_status"] == "vol_churn").mean()
    avg_invol = (master_train_df["churn_status"] == "invol_churn").mean()
    avg_total = avg_vol + avg_invol
    n_total   = len(master_train_df)

    print(f"\n   📊 Global baseline  (N = {n_total:,})")
    print(f"      vol={avg_vol:.1%}  |  invol={avg_invol:.1%}  |  total churn={avg_total:.1%}")

    # Early exit for confident non-churn
    if pred_label == "not_churned" and prob_dict["not_churned"] >= 0.55:
        print(f"\n   ✅  High confidence: NOT CHURNED. No risk factors raised.\n")
        print(SEP + "\n")
        return pred_label, prob_dict

    # ── 5. Helpers ────────────────────────────────────────────────────────────
    def _stats(subset):
        """Return (n, vol_rate, invol_rate) for a filtered subset of master_train."""
        n = len(subset)
        if n == 0:
            return 0, 0.0, 0.0
        v = (subset["churn_status"] == "vol_churn").mean()
        i = (subset["churn_status"] == "invol_churn").mean()
        return n, v, i

    def _flag_parts(vol, invol, avg_v=avg_vol, avg_i=avg_invol, thr=deviation_threshold):
        """Return (list_of_flag_strings, severity_score)."""
        parts, sev = [], 0.0
        dv, di = vol - avg_v, invol - avg_i
        if abs(dv) >= thr:
            direction = "↑ higher" if dv > 0 else "↓ lower"
            parts.append(f"vol churn  {dv:+.1%} vs avg  ({vol:.1%})  {direction}")
            sev = max(sev, abs(dv))
        if abs(di) >= thr:
            direction = "↑ higher" if di > 0 else "↓ lower"
            parts.append(f"invol churn {di:+.1%} vs avg  ({invol:.1%})  {direction}")
            sev = max(sev, abs(di))
        return parts, sev

    def _get(col, default=None):
        """Safe getter from user_raw; returns default for NaN/missing."""
        val = user_raw.get(col, default)
        if val is None:
            return default
        try:
            if pd.isna(val):
                return default
        except TypeError:
            pass
        return val

    risks     = []   # list of (severity, category_label, detail_lines)
    positives = []   # list of strings — protective / neutral findings

    # ── 6. BANK ───────────────────────────────────────────────────────────────
    bank = str(_get("bank_name", "")).strip()
    if bank and bank not in ("nan", "unknown", ""):
        subset = master_train_df[master_train_df["bank_name"] == bank]
        n, bv, bi = _stats(subset)
        if n < min_bank_users:
            # Only flag as uncertain; no severity score
            risks.append((0.001, "🏦 Bank — sparse data", [
                f"Bank : {bank[:55]}",
                f"Only {n} user(s) in training data — below the {min_bank_users}-user threshold.",
                f"Statistics unreliable; cannot draw conclusions from this bank alone.",
            ]))
        else:
            parts, sev = _flag_parts(bv, bi)
            total_bank = bv + bi
            if parts:
                lines = [
                    f"Bank : {bank[:55]}",
                    f"N = {n:,} users  |  total churn = {total_bank:.1%}  "
                    f"(baseline {avg_total:.1%}, Δ {total_bank - avg_total:+.1%})",
                ]
                for p in parts:
                    lines.append(f"  • {p}")
                risks.append((sev, "🏦 Bank", lines))
            elif show_all_features:
                positives.append(f"🏦 Bank '{bank[:35]}' — churn rate near average (N={n:,})")

    # ── 7. 3DS SUPPORT ────────────────────────────────────────────────────────
    tds_raw = _get("card_3d_secure_support")
    if tds_raw is not None:
        tds_str = str(tds_raw).strip()
        # master_train_df has raw string values for this column
        subset = master_train_df[
            master_train_df["card_3d_secure_support"].astype(str).str.strip() == tds_str
        ]
        n, tv, ti = _stats(subset)
        if n >= 30:
            parts, sev = _flag_parts(tv, ti)
            if parts:
                lines = [f"3DS support : '{tds_str}'  (N={n:,})"]
                for p in parts:
                    lines.append(f"  • {p}")
                risks.append((sev, "💳 3DS Support", lines))
            elif show_all_features:
                positives.append(f"💳 3DS support '{tds_str}' — near average (N={n:,})")

    # ── 8. CARD COUNTRY ───────────────────────────────────────────────────────
    card_ctry = str(_get("card_country", "")).strip().lower()
    if card_ctry and card_ctry not in ("nan", "unknown", ""):
        subset = master_train_df[
            master_train_df["card_country"].astype(str).str.strip().str.lower() == card_ctry
        ]
        n, cv, ci = _stats(subset)
        if n >= 30:
            parts, sev = _flag_parts(cv, ci)
            if parts:
                lines = [f"Card country : {card_ctry.upper()}  (N={n:,})"]
                for p in parts:
                    lines.append(f"  • {p}")
                risks.append((sev, "🌍 Card Country", lines))
            elif show_all_features:
                positives.append(f"🌍 Card country {card_ctry.upper()} — near average (N={n:,})")

    # ── 9. BILLING COUNTRY ────────────────────────────────────────────────────
    bill_ctry = str(_get("billing_address_country", "")).strip().lower()
    if bill_ctry and bill_ctry not in ("nan", "unknown", "") and bill_ctry != card_ctry:
        subset = master_train_df[
            master_train_df["billing_address_country"].astype(str).str.strip().str.lower()
            == bill_ctry
        ]
        n, bv2, bi2 = _stats(subset)
        if n >= 30:
            parts, sev = _flag_parts(bv2, bi2)
            if parts:
                lines = [f"Billing country : {bill_ctry.upper()}  (N={n:,})"]
                for p in parts:
                    lines.append(f"  • {p}")
                risks.append((sev, "📮 Billing Country", lines))

    # ── 10. GEO MISMATCH: card ≠ billing ─────────────────────────────────────
    mismatch = float(_get("card_billing_mismatch", 0) or 0)
    if mismatch == 1.0:
        subset = master_train_df[
            master_train_df["card_billing_mismatch"].fillna(0).astype(float) == 1.0
        ]
        n, mv, mi = _stats(subset)
        if n >= 30:
            dv, di = mv - avg_vol, mi - avg_invol
            lines = [
                "Card country ≠ billing country  (geo mismatch flag = 1)",
                f"N = {n:,} users with mismatch",
                f"  • vol churn  {dv:+.1%} vs avg  ({mv:.1%})",
                f"  • invol churn {di:+.1%} vs avg  ({mi:.1%})",
            ]
            risks.append((max(abs(dv), abs(di)), "🗺️  Geo Mismatch", lines))
    elif show_all_features:
        positives.append("🗺️  Billing country = card country — no geo mismatch")

    # ── 11. BANK–CARD COUNTRY MISMATCH ───────────────────────────────────────
    bk_mm = float(_get("bank_card_country_mismatch", 0) or 0)
    if bk_mm == 1.0:
        subset = master_train_df[
            master_train_df["bank_card_country_mismatch"].fillna(0).astype(float) == 1.0
        ]
        n, mv2, mi2 = _stats(subset)
        if n >= 30:
            parts, sev = _flag_parts(mv2, mi2)
            if parts:
                lines = [
                    "Bank country ≠ card country  (bank-card geo mismatch flag = 1)",
                    f"N = {n:,}",
                ]
                for p in parts:
                    lines.append(f"  • {p}")
                risks.append((sev, "🏦🗺️  Bank-Card Mismatch", lines))

    # ── 12. COUNTRY GROUP ────────────────────────────────────────────────────
    cgrp = str(_get("country_group", "")).strip()
    if cgrp and cgrp not in ("nan", "unknown", ""):
        subset = master_train_df[master_train_df["country_group"] == cgrp]
        n, gv, gi = _stats(subset)
        if n >= 30:
            parts, sev = _flag_parts(gv, gi)
            if parts:
                lines = [f"Country group : {cgrp}  (N={n:,})"]
                for p in parts:
                    lines.append(f"  • {p}")
                risks.append((sev, "🌐 Country Group", lines))

    # ── 13. HIGH-RISK CARD (prepaid / virtual) ────────────────────────────────
    high_risk = float(_get("high_risk_card", 0) or 0)
    if high_risk == 1.0:
        subset = master_train_df[
            master_train_df["high_risk_card"].fillna(0).astype(float) == 1.0
        ]
        n, hv, hi = _stats(subset)
        parts, sev = _flag_parts(hv, hi)
        if parts:
            lines = [
                "Card type : Virtual or Prepaid  (high_risk_card = 1)",
                f"N = {n:,} users with virtual/prepaid cards",
            ]
            for p in parts:
                lines.append(f"  • {p}")
            risks.append((sev, "💳 High-Risk Card", lines))
    elif show_all_features:
        positives.append("💳 Standard card (not virtual/prepaid)")

    # ── 14. BUSINESS CARD ────────────────────────────────────────────────────
    is_biz = float(_get("is_business", 0) or 0)
    if is_biz == 1.0:
        subset = master_train_df[
            master_train_df["is_business"].fillna(0).astype(float) == 1.0
        ]
        n, bv3, bi3 = _stats(subset)
        parts, sev = _flag_parts(bv3, bi3)
        if parts:
            lines = [f"Card type : Business  (N={n:,})"]
            for p in parts:
                lines.append(f"  • {p}")
            risks.append((sev, "💼 Business Card", lines))
        elif show_all_features:
            positives.append(f"💼 Business card — churn near average (N={n:,})")

    # ── 15. ALL PAYMENT ATTEMPTS FAILED ──────────────────────────────────────
    all_fail = float(_get("all_attempts_failed", 0) or 0)
    if all_fail == 1.0:
        subset = master_train_df[
            master_train_df["all_attempts_failed"].fillna(0).astype(float) == 1.0
        ]
        n, fv, fi = _stats(subset)
        parts, sev = _flag_parts(fv, fi)
        if parts:
            lines = [
                "ALL payment attempts failed  (failure_rate = 100%)",
                f"N = {n:,} users with total payment failure",
            ]
            for p in parts:
                lines.append(f"  • {p}")
            risks.append((sev, "💸 Total Payment Failure", lines))

    # ── 16. HIGH FAILURE RATE (not 100%, but significant) ────────────────────
    fail_rate = float(_get("failure_rate", 0) or 0)
    if fail_rate > 0.3 and all_fail != 1.0:          # avoid double-reporting
        label_fr = "Extreme (>80%)" if fail_rate > 0.8 else f"High ({fail_rate:.0%})"
        subset_fail = master_train_df[
            master_train_df["failure_rate"].fillna(0) > 0.3
        ]
        n, fv2, fi2 = _stats(subset_fail)
        parts, sev = _flag_parts(fv2, fi2)
        if parts:
            lines = [
                f"Payment failure rate : {fail_rate:.1%}  ({label_fr})",
                f"N = {n:,} users with >30% failure rate",
            ]
            for p in parts:
                lines.append(f"  • {p}")
            risks.append((sev, "⚠️  High Failure Rate", lines))

    # ── 17. 3DS FRICTION ─────────────────────────────────────────────────────
    tds_frict = float(_get("threeds_friction_rate", 0) or 0)
    if tds_frict > 0.2:
        subset_tf = master_train_df[
            master_train_df["threeds_friction_rate"].fillna(0) > 0.2
        ]
        n, tfv, tfi = _stats(subset_tf)
        parts, sev = _flag_parts(tfv, tfi)
        if parts:
            lines = [
                f"3DS friction rate : {tds_frict:.1%}  (bank required 3DS, user didn't authenticate)",
                f"N = {n:,} users with high 3DS friction",
            ]
            for p in parts:
                lines.append(f"  • {p}")
            risks.append((sev, "🔐 3DS Friction", lines))

    # ── 18. ONLY FAILURES — never succeeded ──────────────────────────────────
    only_fail = float(_get("only_failures", 0) or 0)
    if only_fail == 1.0:
        subset = master_train_df[
            master_train_df["only_failures"].fillna(0).astype(float) == 1.0
        ]
        n, ov, oi = _stats(subset)
        parts, sev = _flag_parts(ov, oi)
        if parts:
            lines = [
                "User NEVER had a successful generation  (only_failures = 1)",
                f"N = {n:,} users with zero completed generations",
            ]
            for p in parts:
                lines.append(f"  • {p}")
            risks.append((sev, "❌ Zero Success Rate", lines))

    # ── 19. TIME TO FIRST SUCCESS ────────────────────────────────────────────
    ttfs = float(_get("time_to_first_success_hours", 0) or 0)
    if ttfs > 48 and only_fail != 1.0:
        label_ts = "Never succeeded (full 14-day window)" if ttfs >= 336 else f"{ttfs:.0f} hours"
        subset_slow = master_train_df[
            master_train_df["time_to_first_success_hours"].fillna(0) > 48
        ]
        n, sv, si = _stats(subset_slow)
        parts, sev = _flag_parts(sv, si)
        if parts:
            lines = [
                f"Time to first success : {label_ts}",
                f"N = {n:,} users who took >48h to first success",
            ]
            for p in parts:
                lines.append(f"  • {p}")
            risks.append((sev, "⏱️  Slow Onboarding", lines))
    elif 0 < ttfs <= 1 and show_all_features:
        positives.append(f"⏱️  Fast first success ({ttfs:.1f}h) — strong early engagement signal")

    # ── 20. CREDIT BURN VELOCITY (extreme = top 20%) ─────────────────────────
    cbv = float(_get("gen_credit_burn_vel", 0) or 0)
    if cbv > 0:
        p80_cbv = master_train_df["gen_credit_burn_vel"].quantile(0.80)
        if cbv >= p80_cbv:
            subset_cbv = master_train_df[
                master_train_df["gen_credit_burn_vel"].fillna(0) >= p80_cbv
            ]
            n, xv, xi = _stats(subset_cbv)
            parts, sev = _flag_parts(xv, xi)
            if parts:
                lines = [
                    f"Credit burn velocity : {cbv:.2f}  (top 20%, threshold ≥ {p80_cbv:.2f})",
                    f"N = {n:,} users in extreme-burn tier",
                ]
                for p in parts:
                    lines.append(f"  • {p}")
                risks.append((sev, "🔥 Extreme Credit Burn", lines))

    # ── 21. GENERATION BURST USAGE (extreme = top 20%) ───────────────────────
    gpad = float(_get("gen_per_active_day", 0) or 0)
    if gpad > 0:
        p80_gpad = master_train_df["gen_per_active_day"].quantile(0.80)
        if gpad >= p80_gpad:
            subset_gpad = master_train_df[
                master_train_df["gen_per_active_day"].fillna(0) >= p80_gpad
            ]
            n, gv2, gi2 = _stats(subset_gpad)
            parts, sev = _flag_parts(gv2, gi2)
            if parts:
                lines = [
                    f"Generations per active day : {gpad:.1f}  (top 20%, threshold ≥ {p80_gpad:.1f})",
                    f"N = {n:,} burst-usage users",
                ]
                for p in parts:
                    lines.append(f"  • {p}")
                risks.append((sev, "📈 Extreme Usage Burst", lines))

    # ── 22. FRUSTRATION ──────────────────────────────────────────────────────
    frstr = str(_get("frustration", "")).strip()
    if frstr and frstr not in ("nan", "unknown", ""):
        subset = master_train_df[master_train_df["frustration"] == frstr]
        n, fv3, fi3 = _stats(subset)
        if n >= 30:
            parts, sev = _flag_parts(fv3, fi3)
            if parts:
                lines = [f"Stated frustration : '{frstr}'  (N={n:,})"]
                for p in parts:
                    lines.append(f"  • {p}")
                risks.append((sev, "😤 Frustration Signal", lines))

    # ── 23. SUBSCRIPTION PLAN ────────────────────────────────────────────────
    plan = str(_get("subscription_plan", "")).strip()
    if plan and plan not in ("nan", "unknown", ""):
        subset = master_train_df[master_train_df["subscription_plan"] == plan]
        n, pv, pi = _stats(subset)
        if n >= 30:
            parts, sev = _flag_parts(pv, pi)
            if parts:
                lines = [f"Subscription plan : '{plan}'  (N={n:,})"]
                for p in parts:
                    lines.append(f"  • {p}")
                risks.append((sev, "📋 Subscription Plan", lines))
            elif show_all_features:
                positives.append(f"📋 Plan '{plan}' — churn near average (N={n:,})")

    # ── 24. ENGAGEMENT DROP (week1 vs week2) ─────────────────────────────────
    w1 = float(_get("gen_week1", 0) or 0)
    w2 = float(_get("gen_week2", 0) or 0)
    if w1 > 0:
        drop_pct = (w1 - w2) / max(w1, 1)
        if drop_pct > 0.6:                             # >60% drop from week1 to week2
            subset_drop = master_train_df[
                (master_train_df["gen_week1"].fillna(0) > 0) &
                (
                    (master_train_df["gen_week1"].fillna(0) - master_train_df["gen_week2"].fillna(0))
                    / master_train_df["gen_week1"].fillna(1).clip(lower=1) > 0.6
                )
            ]
            n, dv_v, dv_i = _stats(subset_drop)
            parts, sev = _flag_parts(dv_v, dv_i)
            if n >= 30 and parts:
                lines = [
                    f"Engagement dropped {drop_pct:.0%} week1→week2  "
                    f"({w1:.0f} gens → {w2:.0f} gens)",
                    f"N = {n:,} users with similar steep engagement drop",
                ]
                for p in parts:
                    lines.append(f"  • {p}")
                risks.append((sev, "📉 Engagement Drop", lines))

    # ── 25. ZERO GENERATIONS ─────────────────────────────────────────────────
    has_gen = float(_get("has_generations", 0) or 0)
    gen_total = float(_get("gen_total", 0) or 0)
    if has_gen == 0 or gen_total == 0:
        subset_0 = master_train_df[master_train_df["has_generations"].fillna(0) == 0]
        n, nv, ni = _stats(subset_0)
        if n >= 30:
            parts, sev = _flag_parts(nv, ni)
            if parts:
                lines = [
                    "User made ZERO generations in first 14 days",
                    f"N = {n:,} users with no generation activity",
                ]
                for p in parts:
                    lines.append(f"  • {p}")
                risks.append((sev, "🚫 No Generations", lines))

    # ── Sort by severity (desc) ───────────────────────────────────────────────
    risks.sort(key=lambda x: x[0], reverse=True)

    # ── Print risk factors ────────────────────────────────────────────────────
    print(f"\n{sep}")
    print(f"  🚨 CHURN RISK FACTORS  ({len(risks)} flagged, sorted by severity)")
    print(sep)

    if not risks:
        print(
            "  No individual factors exceeded the deviation threshold.\n"
            "  The model may be leveraging complex feature interactions.\n"
            "  Try lowering deviation_threshold or setting show_all_features=True."
        )
    else:
        for rank, (sev, cat, lines) in enumerate(risks, 1):
            sev_bar = "▓" * int(sev * 20) if sev >= deviation_threshold else ""
            print(f"\n  {rank:>2}. {cat:<35} severity Δ {sev:.1%}  {sev_bar}")
            for line in lines:
                print(f"       {line}")

    # ── Print protective factors ──────────────────────────────────────────────
    if positives:
        print(f"\n{sep}")
        print("  ✅ PROTECTIVE / NEUTRAL FACTORS")
        print(sep)
        for p in positives:
            print(f"  • {p}")

    # ── Summary verdict ───────────────────────────────────────────────────────
    print(f"\n{sep}")
    top = risks[:3] if risks else []
    if pred_label != "not_churned":
        dom_type = (
            "voluntary (user dissatisfaction)"  if prob_dict["vol_churn"] > prob_dict["invol_churn"]
            else "involuntary (payment failure)"
        )
        print(f"  🏁 VERDICT : {pred_label.upper()}  |  dominant driver = {dom_type}")
        if top:
            top_cats = " → ".join(r[1].split()[1] for r in top if len(r[1].split()) > 1)
            print(f"     Top factors: {top_cats if top_cats else 'see above'}")
    else:
        print(f"  🏁 VERDICT : NOT CHURNED  (model is moderately confident)")
    print(SEP + "\n")

    return pred_label, prob_dict

In [ ]:
# ── Usage examples ───────────────────────────────────────────────────────────── 
# explain_user(
#     user_id         = master_train["user_id"].iloc[4],
#     master_train_df = master_train,
#     master_test_df  = master_test,
#     model_lgb       = final_lgb,
#     model_xgb       = final_xgb,
#     X_tr_final      = X_train_final,
#     X_te_final      = X_test_final,
#     label_encoder   = le_target,
# )

explain_user(
    user_id         = test["users"]["user_id"].iloc[0],
    master_train_df = master_train,
    master_test_df  = master_test,
    model_lgb       = final_lgb,
    model_xgb       = final_xgb,
    X_tr_final      = X_train_final,
    X_te_final      = X_test_final,
    label_encoder   = le_target,
    show_all_features = True,    # show protective factors too
    deviation_threshold = 0.03,  # lower threshold = more verbose
)

# for cls in ['vol_churn', 'invol_churn']:
#     uid = master_train[master_train['churn_status'] == cls]['user_id'].iloc[0]
#     explain_user(uid, master_train, master_test,
#                  final_lgb, final_xgb, X_train_final, X_test_final, le_target)

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt
import matplotlib.patches as patches

dt_viz = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=42)
dt_viz.fit(X_train_final, y_all)

# 2. Настройка фигуры
fig, ax = plt.subplots(figsize=(24, 12), facecolor='none') # Прозрачный фон для вставки в слайды

# 3. Отрисовка дерева
out = plot_tree(
    dt_viz,
    feature_names=feat_cols,
    class_names=le_target.classes_,
    filled=True,
    rounded=True,
    precision=2,
    fontsize=14,
    ax=ax,
    proportion=True,
    impurity=False  # Убираем индекс Джини для чистоты
)

# 4. Применяем твой кастомный цвет #d1ff1a ко всем узлам
for artist in out:
    if isinstance(artist, patches.FancyBboxPatch):
        artist.set_facecolor('#d1ff1a')
        artist.set_edgecolor('black')
        artist.set_linewidth(1.5)
    elif isinstance(artist, plt.Text):
        artist.set_color('black') # Текст будет черным для контраста с ярким фоном

plt.title("CHURN DECISION PATHWAY (Categorization Logic)", 
          fontsize=25, fontweight='bold', color='#d1ff1a', pad=20)

plt.tight_layout()
plt.show()
